# 3. Machine Learning Model - Tree-Based Phishing Detection

Building on the rule-based foundation from notebook 2, we now develop ML models to catch the remaining 11.8% of sophisticated phishing sites.

## 1. Foundation & Context

### 1.1 Business Requirement Review

**The ML value proposition:**

Rule-based system caught 88.2% of phishing with perfect precision. The remaining 11.8% (11,944 phishing) have weak individual signals:

- **HasSocialNet=0**: Catches 98.5% of remaining phishing BUT only 78% precision → would block 27,704 legitimate sites
- **Robots=0**: High correlation BUT only 85% precision individually  
- **IsResponsive=0**: Similar story

**The cake and eating it too:**

ML can COMBINE these weak signals to achieve ~100% precision:
- HasSocialNet=0 + Robots=0 + IsResponsive=0 + NoOfImage < 5 + HasExternalFormSubmit=1
- Each feature alone: 70-85% precision
- Combined intelligently: ~100% precision

This is what rules cannot do - they evaluate features in isolation.

**Business outcome:**
- Catch sophisticated phishing without blocking legitimate sites
- Maintain explainability (which feature combination drove the score)

### 1.2 Dataset Reality Check

**Building from dataset4 patterns:**

We're training ML models on dataset4 (235,795 URLs) - the same dataset explored in notebooks 1-2. We believe this approach COULD be highly effective because:
- Dataset captures real phishing patterns (weak signals like HasSocialNet=0, Robots=0, IsResponsive=0)
- Tree models can learn feature combinations that achieve high precision
- 42.8% phishing ratio provides balanced training data

**Acknowledged limitation:**

If phishing methods change significantly (new attack patterns, different evasion techniques), model performance may degrade. This model is trained on current patterns observed in dataset4.

**Our goal:** Build an effective phishing detector by proving ML can combine weak signals to achieve ~100% precision - catching sophisticated phishing without blocking legitimate sites.

## 2. Data Preparation

### 2.1 Load Full Dataset

We load the complete dataset4 (235,795 URLs) with all 56 features. 

**Feature exclusions:**
- **URLSimilarityIndex**: Data leakage (all legitimate sites = 100.0)
- **FILENAME, URL**: Identifiers, not predictive features
- **Domain, TLD, Title**: Text features (need encoding for ML)
- **label**: Target variable

This leaves **49 numeric features** for ML training.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load dataset4
df = pd.read_csv('data/dataset4.csv')

# NOTE: label = 0 is Phishing, label = 1 is Legitimate
print(f"Total URLs: {len(df):,}")
print(f"Phishing: {(df['label']==0).sum():,} ({(df['label']==0).sum()/len(df)*100:.1f}%)")
print(f"Legitimate: {(df['label']==1).sum():,} ({(df['label']==1).sum()/len(df)*100:.1f}%)")

# Exclude problematic features
# - URLSimilarityIndex: Data leakage (all legitimate = 100.0)
# - FILENAME, URL, Domain, TLD, Title: Text features (not numeric)
# - label: Target variable
features_to_exclude = ['URLSimilarityIndex', 'FILENAME', 'URL', 'Domain', 'TLD', 'Title', 'label']
feature_cols = [col for col in df.columns if col not in features_to_exclude]

print(f"\nUsing {len(feature_cols)} features for ML")
print(f"Features: {feature_cols}")

### 2.2 Train/Test Split

**Why ML requires splitting:**

Unlike rules (which we design), ML models LEARN patterns from data. Testing on training data would let the model MEMORIZE instead of learning real patterns. We split to validate the model learned generalizable patterns.

**Our approach - dual evaluation:**

1. **Standard 80/20 split**: Measures general ML performance across all phishing types
2. **Targeted evaluation**: Later test specifically on the 11,944 sophisticated phishing that rules missed

This way we know:
- Does ML work generally? (80/20 test)
- Does ML solve our specific problem? (11,944 test)

In [ ]:
# Split features and target
X = df[feature_cols]
y = df['label']

# 80/20 train/test split, stratified by label
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"Training set: {len(X_train):,} URLs")
print(f"  Phishing: {(y_train==0).sum():,} ({(y_train==0).sum()/len(y_train)*100:.1f}%)")
print(f"  Legitimate: {(y_train==1).sum():,} ({(y_train==1).sum()/len(y_train)*100:.1f}%)")

print(f"\nTest set: {len(X_test):,} URLs")
print(f"  Phishing: {(y_test==0).sum():,} ({(y_test==0).sum()/len(y_test)*100:.1f}%)")
print(f"  Legitimate: {(y_test==1).sum():,} ({(y_test==1).sum()/len(y_test)*100:.1f}%)")

## 3. Model Development

**Do models need our rules?**

No! ML models learn their OWN patterns from the data. We feed them the 52 features, they figure out which combinations matter. Our 7 rules from notebook 2 aren't coded in - the model will RE-DISCOVER them (and more) automatically.

**Model options (simple to complex):**

**1. Logistic Regression** (simplest)
- Learns: "Each feature contributes X points to phishing score"
- Example: `IsHTTPS=0 adds +50 points, HasSocialNet=0 adds +30 points`
- Limitation: Assumes features work independently (can't learn "IF HasSocialNet=0 AND Robots=0 THEN...")

**2. Random Forest** (tree-based)
- Learns: Decision tree rules automatically
- Example: 
  ```
  IF IsHTTPS=0 THEN phishing
  ELSE IF HasSocialNet=0 AND Robots=0 AND NoOfImage < 5 THEN phishing
  ```
- Builds 100+ trees, averages their predictions
- **Why better**: Learns feature COMBINATIONS (the "cake and eating it too" from Section 1.1)

**3. XGBoost** (advanced tree-based)
- Same idea as Random Forest but smarter:
  - Each new tree focuses on fixing previous trees' mistakes
  - Better at finding subtle patterns
- Usually 2-5% more accurate than Random Forest

**Our choice: Random Forest**

We skip Logistic Regression and go straight to Random Forest because:
- Our data has clear feature interactions (HasSocialNet=0 + Robots=0 + IsResponsive=0)
- Tree models automatically learn these combinations
- Will re-discover our 7 rules PLUS find new combinations for sophisticated phishing

### 3.1 Random Forest Model

We train a Random Forest with 100 trees on our training data (188,636 URLs) and evaluate on the test set (47,159 URLs).

**Key metrics we care about:**
- **Precision for Phishing (label=0)**: How many flagged phishing are actually phishing? (minimize false positives)
- **Recall for Phishing (label=0)**: How many actual phishing did we catch?
- **Feature Importance**: Did the model rediscover our 7 rules? Which features matter most?

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Train Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,      # 100 trees
    random_state=42,
    n_jobs=-1              # Use all CPU cores
)

print("Training Random Forest...")
rf_model.fit(X_train, y_train)

# Predict on test set
y_pred = rf_model.predict(X_test)

# Overall accuracy
print(f"\nAccuracy: {accuracy_score(y_test, y_pred)*100:.2f}%")

# Confusion matrix
print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(f"\nTrue Negatives (Phishing correctly identified): {cm[0][0]:,}")
print(f"False Positives (Legitimate flagged as phishing): {cm[1][0]:,}")
print(f"False Negatives (Phishing missed): {cm[0][1]:,}")
print(f"True Positives (Legitimate correctly identified): {cm[1][1]:,}")

# Precision, Recall, F1
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Phishing', 'Legitimate']))

In [ ]:
# Feature Importance - did the model rediscover our rules?
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 15 Most Important Features:")
print(feature_importance.head(15).to_string(index=False))

print("\n\nFeatures from our 7 perfect rules:")
rule_features = ['IsHTTPS', 'NoOfJS', 'NoOfCSS', 'NoOfImage', 'IsDomainIP', 
                 'HasTitle', 'HasFavicon', 'HasDescription', 'HasCopyrightInfo',
                 'NoOfExternalRef', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfSubDomain', 'URLLength']
print(feature_importance[feature_importance['feature'].isin(rule_features)].to_string(index=False))

### 3.2 Edge Case Analysis - The 5 Missed Phishing

Random Forest missed 5 phishing sites out of 20,189 (99.98% recall). Let's analyze what makes these edge cases so hard to detect - even for ML combining weak signals.

In [ ]:
# Find the 5 missed phishing
missed_mask = (y_test == 0) & (y_pred == 1)  # Actually phishing, predicted legitimate
missed_indices = y_test[missed_mask].index

print(f"Found {len(missed_indices)} missed phishing sites:\n")

for idx in missed_indices:
    row = df.loc[idx]
    print(f"URL: {row['URL']}")
    
    # Check against our 7 rules
    caught = False
    if row['NoOfJS'] == 0 and row['NoOfCSS'] == 0 and row['NoOfImage'] == 0:
        print("  Would be CAUGHT by Rule 1: Zero Resources")
        caught = True
    elif row['IsHTTPS'] == 0:
        print("  Would be CAUGHT by Rule 2: No HTTPS")
        caught = True
    elif row['IsDomainIP'] == 1:
        print("  Would be CAUGHT by Rule 3: Domain is IP")
        caught = True
    elif (row['HasTitle'] == 0 and row['HasFavicon'] == 0 and 
          row['HasDescription'] == 0 and row['HasCopyrightInfo'] == 0):
        print("  Would be CAUGHT by Rule 4: Zero Trust Signals")
        caught = True
    elif (row['NoOfExternalRef'] == 0 and row['NoOfSelfRef'] == 0 and row['NoOfEmptyRef'] == 0):
        print("  Would be CAUGHT by Rule 5: No References")
        caught = True
    elif row['NoOfSubDomain'] >= 5:
        print("  Would be CAUGHT by Rule 6: NoOfSubDomain >= 5")
        caught = True
    elif row['URLLength'] > 57:
        print("  Would be CAUGHT by Rule 7: URLLength > 57")
        caught = True
    else:
        print("  NOT caught by any rule - sophisticated phishing")
    
    print(f"  HTTPS: {row['IsHTTPS']} | Resources: JS={row['NoOfJS']}, CSS={row['NoOfCSS']}, Img={row['NoOfImage']}")
    print(f"  Trust: Title={row['HasTitle']}, Favicon={row['HasFavicon']}, Copyright={row['HasCopyrightInfo']}")
    print(f"  Professional: Social={row['HasSocialNet']}, Robots={row['Robots']}, Responsive={row['IsResponsive']}")
    print(f"  Code: {row['LineOfCode']} lines | URL Length: {row['URLLength']} chars\n")

**Key findings about these 5 edge cases:**

**All 5 are sophisticated phishing - NOT caught by any of our 7 rules:**
- All have **HTTPS** (secure connection)
- All have **resources** (JS, CSS, images - invested effort)
- All have **trust signals** (title, favicon, some have copyright/description)
- All have **references** (self-refs, external refs - complex pages)
- **Normal URL lengths** (24-33 chars, not suspicious)

**Sophisticated tactics observed:**
- **2x blogspot.com** (Google's platform - abusing trust)
- **rariblies.blogspot.com** - typo-squatting "Rarible" NFT marketplace
- **000webhostapp.com** - free hosting platform
- **.xyz and .top domains** - modern TLDs often used for phishing

**Why ML missed them:**

These sites invested heavily to look legitimate. Their feature combinations fall in the grey zone between legitimate and phishing. Even combining weak signals across 49 features, Random Forest couldn't achieve 100% certainty without risking false positives.

**The 99.98% recall (20,184/20,189) is remarkable** - these 5 represent the absolute hardest edge cases in the dataset.

### 3.3 XGBoost - Can We Catch the Final 5?

XGBoost is more advanced than Random Forest - each tree focuses on fixing previous mistakes. Let's see if it can catch those 5 edge cases.

In [ ]:
from xgboost import XGBClassifier

# Train XGBoost
xgb_model = XGBClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)

print("Training XGBoost...")
xgb_model.fit(X_train, y_train)

# Predict on test set
y_pred_xgb = xgb_model.predict(X_test)

# Overall accuracy
print(f"\nAccuracy: {accuracy_score(y_test, y_pred_xgb)*100:.2f}%")

# Confusion matrix
print("\nConfusion Matrix:")
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
print(cm_xgb)
print(f"\nTrue Negatives (Phishing correctly identified): {cm_xgb[0][0]:,}")
print(f"False Positives (Legitimate flagged as phishing): {cm_xgb[1][0]:,}")
print(f"False Negatives (Phishing missed): {cm_xgb[0][1]:,}")
print(f"True Positives (Legitimate correctly identified): {cm_xgb[1][1]:,}")

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb, target_names=['Phishing', 'Legitimate']))

# Compare with Random Forest
print("\n" + "="*80)
print("COMPARISON: XGBoost vs Random Forest")
print("="*80)
print(f"Random Forest: Missed {cm[0][1]} phishing, {cm[1][0]} false positives")
print(f"XGBoost:       Missed {cm_xgb[0][1]} phishing, {cm_xgb[1][0]} false positives")

if cm_xgb[0][1] < cm[0][1]:
    print(f"\n✓ XGBoost WINS! Caught {cm[0][1] - cm_xgb[0][1]} more phishing")
elif cm_xgb[0][1] == cm[0][1]:
    print(f"\n= TIE: Both models perform equally")
else:
    print(f"\n✗ Random Forest better: XGBoost missed {cm_xgb[0][1] - cm[0][1]} more")

#### 3.3.1 The One That Got Away

XGBoost caught 4 of the 5 edge cases that Random Forest missed. Let's identify the single remaining phishing site that even XGBoost couldn't catch.

In [ ]:
# Find the 1 phishing XGBoost missed
xgb_missed_mask = (y_test == 0) & (y_pred_xgb == 1)
xgb_missed_indices = y_test[xgb_missed_mask].index

print(f"XGBoost missed {len(xgb_missed_indices)} phishing site:\n")

for idx in xgb_missed_indices:
    row = df.loc[idx]
    print(f"URL: {row['URL']}")
    print(f"Domain: {row['Domain']}")
    print(f"TLD: {row['TLD']}")
    
    print(f"\nKey features:")
    print(f"  HTTPS: {row['IsHTTPS']} | Resources: JS={row['NoOfJS']}, CSS={row['NoOfCSS']}, Img={row['NoOfImage']}")
    print(f"  LineOfCode: {row['LineOfCode']}")
    print(f"  Trust signals: Title={row['HasTitle']}, Favicon={row['HasFavicon']}, Description={row['HasDescription']}, Copyright={row['HasCopyrightInfo']}")
    print(f"  Professional: Social={row['HasSocialNet']}, Robots={row['Robots']}, Responsive={row['IsResponsive']}")
    print(f"  References: Self={row['NoOfSelfRef']}, External={row['NoOfExternalRef']}, Empty={row['NoOfEmptyRef']}")
    print(f"  URL: Length={row['URLLength']}, Subdomains={row['NoOfSubDomain']}")
    
    print(f"\nThis is the ultimate edge case - even XGBoost's advanced learning couldn't distinguish it from legitimate sites.")